<img src="../img/GTK_Logo_Social_Icon.jpg" width=175 align="right" />

# Worksheet 12.1: Building & Using an MCP Server - Answers

In Module 11 you hand-wrote tools and bound them directly to one agent. The
**Model Context Protocol (MCP)** lets tools live in a separate *server* that any
MCP-aware application can connect to.

In this lab you connect an agent to a small security-tools MCP server, load its
tools over MCP, and let the agent investigate emails. The server
(`mcp_server.py`) is given to you; your job is the **client side**: connecting,
loading the tools, and wiring them into an agent.

Docs: [langchain-mcp-adapters](https://github.com/langchain-ai/langchain-mcp-adapters) · [Model Context Protocol](https://modelcontextprotocol.io/)


In [ ]:
# --- data-loader (local files / Databricks volume) ---
import os
import tempfile
try:
    dbutils  # defined only on Databricks
    DATA_HOME = os.environ.get("COURSE_DATA_HOME", "/Volumes/ai_cyber_bootcamp/course/data/")
    WORK_HOME = os.path.abspath("../work") + "/"
    SCRATCH_HOME = tempfile.gettempdir()
    ON_DATABRICKS = True
except NameError:
    DATA_HOME = "../data/"
    WORK_HOME = DATA_HOME
    SCRATCH_HOME = WORK_HOME
    ON_DATABRICKS = False
os.makedirs(WORK_HOME, exist_ok=True)

def handoff(name):
    """Output of an earlier worksheet: the student's own copy if it exists, else the shipped one."""
    p = WORK_HOME + name
    return p if os.path.exists(p) else DATA_HOME + name

COURSE_LLM_MODEL = (
    os.environ.get("COURSE_LLM_ENDPOINT", "databricks-deepseek-v4-flash-0731")
    if ON_DATABRICKS
    else os.environ.get("COURSE_LLM_MODEL", "claude-opus-4-8")
)

# Vision calls: the default endpoint above is text-only, so image cells pass this
# model explicitly to make_client()'s caller instead of COURSE_LLM_MODEL.
COURSE_LLM_VISION_MODEL = (
    os.environ.get("COURSE_LLM_VISION_ENDPOINT", "databricks-glm-5-3-flash")
    if ON_DATABRICKS
    else os.environ.get("COURSE_LLM_MODEL", "claude-opus-4-8")
)

# Module 11.2 reads one ai_examples skill file. ai_examples/ itself is out of scope
# for the port, so the volume ships only that one file, under DATA_HOME/skills/;
# locally the file is still at its original repo path.
SKILLS_HOME = DATA_HOME + "skills/" if ON_DATABRICKS else "../ai_examples/skills/"


def make_client():
    """OpenAI-compatible chat client: Anthropic locally, the workspace endpoint on Databricks."""
    from openai import OpenAI
    if ON_DATABRICKS:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        return OpenAI(api_key=ctx.apiToken().get(),
                      base_url=f"{ctx.apiUrl().get()}/serving-endpoints")
    from dotenv import load_dotenv
    load_dotenv()
    return OpenAI(api_key=os.environ["ANTHROPIC_API_KEY"],
                  base_url="https://api.anthropic.com/v1/")


def make_llm(**kwargs):
    """LangChain chat model: ChatDatabricks on Databricks, ChatAnthropic locally."""
    if ON_DATABRICKS:
        from databricks_langchain import ChatDatabricks
        return ChatDatabricks(endpoint=COURSE_LLM_MODEL, **kwargs)
    from langchain_anthropic import ChatAnthropic
    return ChatAnthropic(model=COURSE_LLM_MODEL, **kwargs)
# --- end data-loader ---


In [ ]:
# Load Libraries - Make sure to run this cell!
import os
import sys
from dotenv import load_dotenv
# the chat model comes from make_llm() in the data-loader cell
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

# Loads ANTHROPIC_API_KEY from your .env file in the project root.
load_dotenv()

MODEL = COURSE_LLM_MODEL

# Find mcp_server.py whether this notebook runs from notebooks/, answers/,
# or the repo root, so the path never depends on your working directory.
SERVER_PATH = next(
    (os.path.abspath(p) for p in
     ("mcp_server.py", "../notebooks/mcp_server.py", "notebooks/mcp_server.py")
     if os.path.exists(p)), None)
assert SERVER_PATH, ("Could not find mcp_server.py. Locally, run this from "
                     "notebooks/ or answers/. On Databricks it lives beside "
                     "the course notebooks, one directory up from this one.")

## The server you'll connect to

Open **`mcp_server.py`** (in this folder) and skim it. It is a complete MCP
server built with `FastMCP`, and it exposes three simulated security tools:

- `check_url_reputation` — reputation of a URL or domain
- `check_sender_reputation` — reputation of a sender address
- `check_email_authentication` — SPF / DKIM / DMARC analysis

You do not need to edit the server. An MCP server is just a program that exposes
tools over a standard protocol; the host launches it and your agent calls its
tools without knowing how they are implemented.


## Step 1 — Connect a client to the server

`MultiServerMCPClient` takes a dictionary describing each server: the command to
launch it, its arguments, and the transport. Because the server runs locally, we
launch it as a subprocess and talk to it over **stdio**.


In [ ]:
server_config = {
    "security-tools": {
        "command": sys.executable,   # the same Python running this notebook
        "args": [SERVER_PATH],       # the server file (resolved in the setup cell)
        "transport": "stdio",
    }
}

client = MultiServerMCPClient(server_config)
print("Client configured for the security-tools server.")

## Step 2 — Load the server's tools

`get_tools()` asks the server what it offers and hands the tools back as
LangChain tools that your agent can call. It is asynchronous, so we `await` it.


In [ ]:
# Windows users, If you running into errors as this is trying to output to stderr, uncomment the lines below as a workaround
# import ipykernel.iostream
# _mcp_log_file = open("mcp_stderr.log", "w")
# ipykernel.iostream.OutStream.fileno = lambda self: _mcp_log_file.fileno()

tools = await client.get_tools()

for t in tools:
    print(f"{t.name}: {t.description.splitlines()[0]}")

assert len(tools) == 3, "Expected 3 tools from the security-tools server"

## Step 3 — Build an agent with those tools

Hand the tools to `create_agent`, exactly as in Module 12. The agent does not
care that these tools live in a separate MCP server rather than in this notebook.


In [ ]:
SYSTEM_PROMPT = (
    "You are a SOC phishing-triage assistant. Investigate the email using the "
    "available tools, then give a final verdict of 'phishing', 'suspicious', or "
    "'legitimate' with one or two sentences of reasoning."
)

llm = make_llm()

agent = create_agent(llm, tools, system_prompt=SYSTEM_PROMPT)

## Step 4 — Investigate a phishing email

Send the agent an email and watch it decide which tools to call. It should reach
a `phishing` verdict on its own.


In [ ]:
PHISHING_EMAIL = """From: PayPal Billing <billing@paypa1-secure.com>
Authentication results: SPF=fail, DKIM=none, DMARC=fail

Your account has been locked for suspicious activity. Verify your identity
immediately at http://paypa1-secure.com/login or your account will be closed.
"""

result = await agent.ainvoke({"messages": [("user", PHISHING_EMAIL)]})
print(result["messages"][-1].content)

## Step 5 — Now a legitimate email

Run the same investigation on a real message. The agent should call the tools,
see clean authentication and a trusted domain, and return `legitimate`.


In [ ]:
LEGIT_EMAIL = """From: GitHub <noreply@github.com>
Authentication results: SPF=pass, DKIM=pass, DMARC=pass

A new sign-in to your account from Chrome on macOS. If this was you, no action
is needed. You can review activity at https://github.com/settings/security
"""

result = await agent.ainvoke({"messages": [("user", LEGIT_EMAIL)]})
print(result["messages"][-1].content)

## What you built

You connected an agent to tools it does not contain. The same agent pattern from
Module 13 now uses tools that live in a separate, swappable MCP server. Point the
client at a different server and the agent gains new abilities with no change to
its own code.

**Security reminder:** every MCP server you connect is code you are trusting with
your agent's context and actions. Vet and pin the servers you use, give each one
the least access it needs, and keep a human in the loop for anything destructive.
